In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

# ==========================================
# LOAD DATASET
# ==========================================

df = pd.read_csv(
    "../datasets/multi_state_final_dataset.csv"
)

print("INITIAL DATA:")
print(df.head())

print("\nTOTAL ROWS:")
print(len(df))

# ==========================================
# CLEAN STRINGS
# ==========================================

string_cols = [
    'State_Name',
    'District_Name',
    'Crop'
]

for col in string_cols:

    df[col] = (

        df[col]

        .astype(str)

        .str.strip()

        .str.lower()

    )

# ==========================================
# SORT FOR TEMPORAL FEATURES
# ==========================================

df = df.sort_values(
    by=[
        'State_Name',
        'District_Name',
        'Crop',
        'Year'
    ]
)

# ==========================================
# LABEL ENCODING
# ==========================================

crop_encoder = LabelEncoder()
state_encoder = LabelEncoder()
district_encoder = LabelEncoder()

df['Crop_Encoded'] = crop_encoder.fit_transform(
    df['Crop']
)

df['State_Encoded'] = state_encoder.fit_transform(
    df['State_Name']
)

df['District_Encoded'] = district_encoder.fit_transform(
    df['District_Name']
)

# ==========================================
# TEMPERATURE ANOMALY
# ==========================================

df['Temp_Anomaly'] = (

    df['T2M']

    - df.groupby('District_Name')['T2M']
      .transform('mean')

)

# ==========================================
# RAINFALL ANOMALY
# ==========================================

df['Rainfall_Anomaly'] = (

    df['PRECTOTCORR']

    - df.groupby('District_Name')['PRECTOTCORR']
      .transform('mean')

)

# ==========================================
# HUMIDITY ANOMALY
# ==========================================

df['Humidity_Anomaly'] = (

    df['RH2M']

    - df.groupby('District_Name')['RH2M']
      .transform('mean')

)

# ==========================================
# PREVIOUS YEAR YIELD
# ==========================================

df['Previous_Yield'] = (

    df.groupby([
        'District_Name',
        'Crop'
    ])['Yield']

    .shift(1)

)

# ==========================================
# 3-YEAR ROLLING AVERAGE
# ==========================================

df['Rolling_Yield_Mean'] = (

    df.groupby([
        'District_Name',
        'Crop'
    ])['Yield']

    .transform(

        lambda x:

        x.rolling(
            window=3,
            min_periods=1
        ).mean()

    )

)

# ==========================================
# YIELD CHANGE %
# ==========================================

df['Yield_Change_Percent'] = (

    df.groupby([
        'District_Name',
        'Crop'
    ])['Yield']

    .pct_change()

)

# ==========================================
# NDVI TREND
# ==========================================

df['NDVI_Trend'] = (

    df.groupby([
        'District_Name',
        'Crop'
    ])['NDVI']

    .diff()

)

# ==========================================
# HANDLE NULLS
# ==========================================

df = df.fillna(0)

# ==========================================
# REMOVE EXTREME VALUES
# ==========================================

df = df.replace(
    [np.inf, -np.inf],
    0
)

# ==========================================
# SAVE FINAL FEATURE DATASET
# ==========================================

df.to_csv(

    "../datasets/feature_engineered_multi_state_dataset.csv",

    index=False

)

print("\nFEATURE ENGINEERING COMPLETE!")

print(df.head())

print("\nFINAL SHAPE:")
print(df.shape)

print("\nFINAL COLUMNS:")
print(df.columns)

INITIAL DATA:
      State_Name District_Name  Year        T2M       RH2M  PRECTOTCORR  \
0  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
1  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
2  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
3  uttar pradesh          agra  2015  26.391288  43.295863       589.35   
4  uttar pradesh          agra  2015  26.391288  43.295863       589.35   

       NDVI          Crop     Yield  
0  0.402542         bajra  1.400917  
1  0.402542  cotton(lint)  0.203390  
2  0.402542         jowar  0.943201  
3  0.402542         maize  1.797297  
4  0.402542          rice  2.266405  

TOTAL ROWS:
20414

FEATURE ENGINEERING COMPLETE!
    State_Name District_Name  Year        T2M       RH2M  PRECTOTCORR  \
324  rajasthan         ajmer  2017  25.556000  41.104849       509.92   
404  rajasthan         ajmer  2020  25.280710  46.572951       512.03   
269  rajasthan         ajmer  2015  25